# Iskanje podatkov PlanetScope in ustvarjanje naročila

Poiščimo satelitske posnetke izbranega območja in ustvarimo naročilo. Naročilo shranimo v zbirko (collection) in ga pošljemo v obdelavo.

## Zahteve

- nameščen Python
- knjižnica requests
- [Planet Account](https://www.planet.com/account/#/)
- API ključ za dostop do Planet API

In [ ]:
# Potrebne knjižnice
import os
import json
import math
import requests
from typing import Dict, List, Iterator

## Nastavitev API ključa

Najprej moramo nastaviti naš API ključ, da bomo lahko dostopali do Planet API. To lahko storimo tako, da ga shranimo kot spremenljivko okolja ali pa ga neposredno uporabimo v našem Python skriptu.

In [ ]:
# Če vaš Planet API ključ ni nastavljen kot spremenljivka okolja, ga lahko prilepite spodaj
if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = 'PASTE_YOUR_API_KEY_HERE'

In [ ]:
# Print API key to verify it was loaded correctly (optional)
print(f'API Key Loaded: {API_KEY[:4]}...{API_KEY[-4:]}')

## Določitev območja interesa

Območje interesa (**Area of Interest** ali *AOI*) določa geografsko območje, iz katerega želimo pridobiti podatke.

Pri Data API je to lahko preprost pravokotnik (bounding box) s štirimi vogali ali bolj kompleksna oblika, dokler je definicija zapisana v formatu [GeoJSON](http://geojson.org/).

V tem primeru bomo uporabili preprost pravokotnik. Za enostavno ustvarjanje lahko uporabimo [geojson.io](http://geojson.io/), kjer hitro narišemo obliko in dobimo GeoJSON zapis:

![geojsonio.png](./slike/geojsonio.png)

Za zahtevo Data API potrebujemo samo objekt **geometry**, ki ga lahko preberemo iz GeoJSON datoteke. Ta objekt vsebuje koordinate, ki definirajo naše AOI. Na primer, če imamo GeoJSON datoteko z imenom `aoi.geojson`, lahko preberemo geometrijo.

In [ ]:
# Ime datoteke z AOI
aoi_file = './podatki/kranj_aoi.geojson'
with open(aoi_file) as f:
    geojson_data = json.load(f)
geojson_geometry = geojson_data['features'][0]['geometry']

In [ ]:
# Izpis geometrije AOI, da preverite, ali je pravilno prebrana
print(json.dumps(geojson_geometry, indent=2, ensure_ascii=False))

## Nastavitve

In [ ]:
# Nastavitve iskanja in naročila
ORDER_NAME = "Kranj 2023"  # Poimenujte naročilo po želji
ITEM_TYPE = "PSScene"
PRODUCT_BUNDLE = "analytic_udm2"   # change if your setup needs another bundle
START_DATE = "2023-01-01T00:00:00Z"
END_DATE = "2023-12-31T23:59:59Z"
MAX_CLOUD_COVER = 0.20             # 20%
BATCH_SIZE = 500                   # Orders API supports up to 500 items/order

In [ ]:
# API endpointa za iskanje posnetkov in ustvarjanje naročil
DATA_SEARCH_URL = "https://api.planet.com/data/v1/quick-search"
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"

### API povezave in avtentikacija

V tej sekciji določimo URL-je storitev in inicializiramo HTTP sejo z API ključem.

In [ ]:
# Inicializacija HTTP seje in avtentikacije za vse API klice
SESSION = requests.Session()
SESSION.headers.update({"Content-Type": "application/json"})
SESSION.auth = (API_KEY, "")

## Pomožne funkcije

Funkcije spodaj pripravijo filter iskanja, izvajajo straničeno branje rezultatov in ustvarijo naročila.

In [ ]:
# Sestava kombiniranega filtra za AOI, časovno obdobje in oblačnost
def build_search_filter(aoi: Dict, start_date: str, end_date: str, max_cloud: float) -> Dict:
    # Sestavimo kombinirani filter: geometrija + datum + oblačnost.
    return {
        "type": "AndFilter",
        "config": [
            {
                "type": "GeometryFilter",
                "field_name": "geometry",
                "config": aoi
            },
            {
                "type": "DateRangeFilter",
                "field_name": "acquired",
                "config": {
                    "gte": start_date,
                    "lte": end_date
                }
            },
            {
                "type": "RangeFilter",
                "field_name": "cloud_cover",
                "config": {
                    "lte": max_cloud
                }
            }
        ]
    }

In [ ]:
# Iskanje posnetkov preko Quick Search API z avtomatskim prehajanjem po straneh
def search_items(item_type: str, search_filter: Dict, page_size: int = 250) -> Iterator[Dict]:
    body = {
        "item_types": [item_type],
        "filter": search_filter
    }

    # Začetni klic z naraščajočim časovnim vrstnim redom.
    url = f"{DATA_SEARCH_URL}?_sort=acquired%20asc&_page_size={page_size}"
    resp = SESSION.post(url, json=body)
    resp.raise_for_status()
    data = resp.json()

    # Iteriramo čez vse strani rezultatov, dokler API vrača _next povezavo.
    while True:
        for feature in data.get("features", []):
            yield feature

        next_url = data.get("_links", {}).get("_next")
        if not next_url:
            break

        next_resp = SESSION.get(next_url)
        next_resp.raise_for_status()
        data = next_resp.json()

In [ ]:
# Razdeli seznam ID-jev na manjše pakete velikosti size
def chunked(seq: List[str], size: int) -> Iterator[List[str]]:
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [ ]:
# Ustvarjanje hosted naročila in vračilo API odziva
def create_hosted_order(order_name: str, item_ids: List[str], item_type: str, product_bundle: str) -> Dict:
    # V payload vključimo hosting.sentinel_hub, da Planet ustvari novo kolekcijo.
    payload = {
        "name": order_name,
        "source_type": "scenes",
        "products": [
            {
                "item_ids": item_ids,
                "item_type": item_type,
                "product_bundle": product_bundle
            }
        ],
        "hosting": {
            "sentinel_hub": {
                # omit collection_id -> Planet creates a new data collection
                "create_configuration": True
            }
        }
    }

    resp = SESSION.post(ORDERS_URL, json=payload)
    resp.raise_for_status()
    return resp.json()

## Iskanje posnetkov

In [ ]:
# Filter za iskanje
search_filter = build_search_filter(geojson_geometry, START_DATE, END_DATE, MAX_CLOUD_COVER)

In [ ]:
# Iskanje posnetkov v katalogu in priprava seznama ID-jev
print("Searching Planet catalog...")
# Iz iskalnika preberemo vse najdene feature-je in izluščimo njihove ID-je.
features = list(search_items(ITEM_TYPE, search_filter))
item_ids = [f["id"] for f in features]

In [ ]:
# Izpis skupnega števila najdenih posnetkov
print(f"Found {len(item_ids)} items")

In [ ]:
# optional: print a few matches
for f in features[:5]:
    props = f.get("properties", {})
    print(f"- {f['id']} | acquired={props.get('acquired')} | cloud_cover={props.get('cloud_cover')}")

## Ustvarjanje naročil

Najdene posnetke razdelimo v pakete in za vsak paket ustvarimo ločeno naročilo.

In [ ]:
# create one or more orders if needed
orders = []
num_batches = math.ceil(len(item_ids) / BATCH_SIZE)

In [ ]:
# Ustvarjanje enega ali več naročil glede na velikost batcha
for idx, batch in enumerate(chunked(item_ids, BATCH_SIZE), start=1):
    # Če je rezultatov več kot BATCH_SIZE, jih razdelimo na več naročil.
    batch_name = f"{ORDER_NAME}-part-{idx:02d}-of-{num_batches:02d}" if num_batches > 1 else ORDER_NAME
    print(f"Creating order: {batch_name} ({len(batch)} items)")
    order = create_hosted_order(batch_name, batch, ITEM_TYPE, PRODUCT_BUNDLE)
    orders.append(order)

    order_id = order.get("id")
    state = order.get("state")
    collection_id = (
        order.get("hosting", {})
                .get("sentinel_hub", {})
                .get("collection_id")
    )

    print(f"  order_id: {order_id}")
    print(f"  state: {state}")
    print(f"  collection_id: {collection_id}")

## Povzetek rezultatov

Na koncu izpišemo vsa ustvarjena naročila in njihovo trenutno stanje.

In [ ]:
# Končni izpis ustvarjenih naročil in njihovih stanj
print("\nDone.")
print("Created orders:")
for order in orders:
    print(f"- {order.get('id')} | {order.get('name')} | {order.get('state')}")